# Configuration model and triangle graphs: test the prediction and compare

Notebook 05 derives a prediction for the configuration **multigraph** with
$K=2\operatorname{Poisson}(\lambda)$ degrees. Notebook 06 collects matched-condition
triangle graphs, CM multigraphs, and their erased simple projections. Here we
read those saved measurements, without generating any new graphs.

We ask three questions, in this order:

1. Does the CM multigraph approach its mathematical prediction as $n$ increases?
2. How much does removing loops and repeated edges change its 3-core?
3. How does the triangle graph differ when the limiting degree law is the same?

Near a discontinuous transition, a mean can average graphs with no large core
and graphs with a substantial one. We therefore show individual-graph variation
as well as means, and inspect distributions around the rise. Lines connect
sampled means; there is no smoothing or fitted transition curve.

In [ ]:
import csv
import json
from collections import defaultdict
from math import exp, factorial
from pathlib import Path

import numpy as np
from IPython.display import display
from matplotlib.figure import Figure

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "notebooks").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"

%run "{NOTEBOOK_DIR}/05_configuration_theory.ipynb"
%matplotlib inline

PROJECT_ROOT = NOTEBOOK_DIR.parent

## Choose saved measurements and figure settings

Use the directory written by notebook 06. `histogram_lambdas` selects actual
sampled parameter values, rather than interpolating between experiments.
`histogram_n=None` uses the largest saved size.

`critical_exclusion` sets a fixed band around the CM critical point that is
excluded **only from the aggregate error table**. Finite-size rounding is
strongest there, so we also inspect the full curves and the closer view. The
band is an analysis choice, not a bound on the finite-size transition window.

`large_core_cutoff` defines an additional descriptive event: at least this
fraction of vertices survives. Its 50% probability crossing gives a coarse
onset bracket on the sampled grid. It does not estimate a universal critical
parameter, and changing the cutoff can change the bracket.

In [ ]:
config = {
    "data_dir": "results/runs/configuration_comparison",  # Written by notebook 06.
    "figure_dir": "results/figures/configuration_comparison",
    "focus_half_width": 0.25,  # Zoom around the theoretical CM critical lambda.
    "degree_lambda": 1.5,  # A sampled value used for the degree-law comparison.
    "histogram_n": None,  # Use the largest size, or choose a saved size explicitly.
    "histogram_lambdas": [1.24, 1.285, 1.35],  # Below, near, and above the CM rise.
    "critical_exclusion": 0.08,  # Fixed band omitted from aggregate error scores.
    "large_core_cutoff": 0.05,  # Descriptive event, not the mathematical definition.
}
config

## Read and summarize independent trials

The CSVs retain the trial seed, the realised degree histogram, and the core
fraction. We check that histogram counts include all vertices and agree with
the recorded mean. A duplicate trial key would accidentally give that graph
extra weight, so it raises an error.

The shaded interval in the curve plots is the **middle half of individual
graphs** (25th to 75th percentile). Error bars show **one standard error of the
mean**, using independent trials. These describe different things: graphs can
remain variable even when their average is measured accurately. With one trial,
a standard error is unavailable and no error bar is drawn.

In [ ]:
MODEL_LABELS = {
    "triangle_rig": "Triangle RIG",
    "cm_multigraph": "CM multigraph",
    "cm_erased": "CM erased",
}
MODEL_COLORS = {"triangle_rig": "C0", "cm_multigraph": "C1", "cm_erased": "C2"}


def load_comparison(directory: str | Path) -> list[dict]:
    """Read saved trials only; fail clearly if simulation data is missing."""
    paths = sorted(Path(directory).glob("comparison_n*.csv"))
    if not paths:
        raise FileNotFoundError(
            f"No comparison_n*.csv files in {directory}. Run notebook 06 first."
        )
    rows, seen = [], set()
    integer_fields = {
        "n",
        "replicate",
        "trial_seed",
        "seed",
        "core_size",
        "loops",
        "parallel_excess",
    }
    for path in paths:
        with path.open(newline="", encoding="utf-8") as stream:
            for saved in csv.DictReader(stream):
                row = {
                    key: value
                    if key in {"model", "degree_histogram"}
                    else int(value)
                    if key in integer_fields
                    else float(value)
                    for key, value in saved.items()
                }
                row["degree_histogram"] = json.loads(row["degree_histogram"])
                key = (row["model"], row["n"], row["lambda"], row["replicate"])
                if row["model"] not in MODEL_LABELS or key in seen:
                    raise ValueError(
                        f"Unknown model or duplicate trial in {path}: {key}"
                    )
                seen.add(key)
                counts = np.asarray(row["degree_histogram"])
                if (
                    row["n"] <= 0
                    or counts.ndim != 1
                    or np.any(counts < 0)
                    or counts.sum() != row["n"]
                ):
                    raise ValueError(f"Invalid degree histogram in {path}: {key}")
                mean = np.arange(len(counts)) @ counts / row["n"]
                if not np.isclose(mean, row["degree_mean"], atol=1e-12):
                    raise ValueError(f"Degree histogram and mean disagree: {key}")
                if not 0 <= row["core_size"] <= row["n"] or not np.isclose(
                    row["core_fraction"], row["core_size"] / row["n"], atol=1e-12
                ):
                    raise ValueError(f"Invalid core measurement: {key}")
                rows.append(row)
    return rows


def sample_statistics(values) -> dict:
    """Describe graph-to-graph spread separately from uncertainty in the mean."""
    values = np.asarray(values, dtype=float)
    return {
        "trials": len(values),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "q25": float(np.percentile(values, 25)),
        "q75": float(np.percentile(values, 75)),
        "minimum": float(np.min(values)),
        "maximum": float(np.max(values)),
        "standard_error": float(np.std(values, ddof=1) / np.sqrt(len(values)))
        if len(values) > 1
        else None,
    }


def summarize_comparison(
    rows: list[dict], large_core_cutoff: float = 0.05
) -> list[dict]:
    """One row for each model, size, and lambda, with degree diagnostics."""
    if not rows or not 0 < large_core_cutoff <= 1:
        raise ValueError("Provide trials and a core cutoff in (0, 1]")
    groups = defaultdict(list)
    for row in rows:
        groups[(row["model"], row["n"], row["lambda"])].append(row)
    summary = []
    for (model, n, lam), group in sorted(groups.items()):
        values = [row["core_fraction"] for row in group]
        result = {"model": model, "n": n, "lambda": lam, **sample_statistics(values)}
        result["large_core_probability"] = float(
            np.mean(np.array(values) >= large_core_cutoff)
        )
        for field in (
            "degree_mean",
            "degree_variance",
            "odd_degree_fraction",
            "simple_transitivity",
            "loops",
            "parallel_excess",
        ):
            result[field] = float(np.mean([row[field] for row in group]))
        summary.append(result)
    return summary


def save_table(rows: list[dict], path: Path):
    """Save an analysis table without altering the raw measurements."""
    if not rows:
        print(f"No rows for {path.name}; no table written.")
        return
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


rows = load_comparison(PROJECT_ROOT / config["data_dir"])
summary = summarize_comparison(rows, config.get("large_core_cutoff", 0.05))
sizes = sorted({row["n"] for row in rows})
lambdas = sorted({row["lambda"] for row in rows})
critical = cm_critical_point()
figure_dir = PROJECT_ROOT / config["figure_dir"]
figure_dir.mkdir(parents=True, exist_ok=True)
figures = {}
print(
    f"Loaded {len(rows):,} measurements; sizes {sizes}; {len(lambdas)} lambda values."
)
print(
    f"CM critical lambda = {critical['lambda']:.8f}, positive-branch density = {critical['rho']:.8f}."
)

## First check: the support equation and the discontinuous branch

Notebook 05 derives $x=F_\lambda(x)$, where $x$ is the probability that a vertex
can support an arriving neighbour using at least two other surviving neighbours.
The trivial solution $x=0$ always exists. At the threshold a positive solution
appears by tangency: $F_\lambda(x)=x$ and $F'_\lambda(x)=1$. Above it, peeling
from full support selects the **largest** root.

The graph below shows the residual rather than an iteration near tangency,
where convergence is slow. The predicted limiting core fraction jumps from
zero to about $0.138$. The value at the critical parameter itself is shown as
the positive branch's right-hand limit. The theorem's usual positive-crossing
condition fails at tangency, so we do not treat it as a deterministic finite-size
prediction at exactly $\lambda_c$.

In [ ]:
figure = Figure(figsize=(8, 4.5), layout="constrained")
axis = figure.subplots()
x_grid = np.linspace(0, 1, 700)
for lam in [critical["lambda"] - 0.05, critical["lambda"], critical["lambda"] + 0.05]:
    (line,) = axis.plot(
        x_grid, cm_message(x_grid, lam) - x_grid, label=rf"$\lambda={lam:.4f}$"
    )
    prediction = cm_prediction(lam, critical=critical)
    axis.scatter(prediction["x"], 0, color=line.get_color(), s=35, zorder=4)
axis.axhline(0, color="black", linewidth=0.8)
axis.set(
    xlabel="Support probability x",
    ylabel=r"$F_\lambda(x)-x$",
    title="The positive branch appears at a tangency",
)
axis.grid(alpha=0.25)
axis.legend(title="Dots mark the selected roots")
figures["fixed_points"] = figure
display(figure)

## Validate the multigraph prediction across sizes

For each sampled $\lambda$ we compare the simulated mean with $\rho(\lambda)$
from notebook 05. Only the retained CM multigraph directly tests this theorem.
The erased version will be checked separately.

The limiting curve has a jump, so we draw its two branches separately. The
open circle is the positive right-hand limit at the critical point; the dotted
line marks $\lambda_c$. A finite graph can have a substantial core just below
that line, or lose it just above. Around the transition, the mean can sit between
the two typical outcomes. This is expected finite-size behaviour, not evidence
that the root finder should follow a different branch.

In [ ]:
# Keep the jump visible: never connect the zero and positive branches diagonally.
theory_lambdas = sorted(
    set(np.linspace(min(lambdas), max(lambdas), 450))
    | set(lambdas)
    | {critical["lambda"]}
)
theory_curve = [cm_prediction(lam, critical=critical) for lam in theory_lambdas]


def draw_theory(axis):
    """Plot separate limiting branches and label their critical right-hand limit."""
    below = [row for row in theory_curve if row["lambda"] < critical["lambda"]]
    above = [row for row in theory_curve if row["lambda"] >= critical["lambda"]]
    if below:
        axis.plot(
            [row["lambda"] for row in below], [0] * len(below), color="black", lw=2
        )
    if above:
        axis.plot(
            [row["lambda"] for row in above],
            [row["rho"] for row in above],
            color="black",
            lw=2,
            label="CM limiting prediction",
        )
    axis.plot(critical["lambda"], critical["rho"], "o", mfc="white", mec="black", ms=6)
    axis.axvline(critical["lambda"], color="black", ls=":", alpha=0.6)


def draw_measurements(axis, groups, color, label, linestyle="-"):
    """Means and their standard errors, with individual-graph quartile shading."""
    groups = sorted(groups, key=lambda row: row["lambda"])
    grid = [row["lambda"] for row in groups]
    axis.fill_between(
        grid,
        [row["q25"] for row in groups],
        [row["q75"] for row in groups],
        color=color,
        alpha=0.12,
    )
    axis.plot(
        grid,
        [row["mean"] for row in groups],
        marker="o",
        ms=3,
        color=color,
        ls=linestyle,
        label=label,
    )
    available = [row for row in groups if row["standard_error"] is not None]
    axis.errorbar(
        [row["lambda"] for row in available],
        [row["mean"] for row in available],
        yerr=[row["standard_error"] for row in available],
        fmt="none",
        ecolor=color,
        capsize=2,
        lw=0.8,
    )


def core_axes(axis):
    axis.set(xlabel=r"$\lambda$", ylabel="3-core fraction", ylim=(-0.01, 1))
    axis.grid(alpha=0.25)


figure = Figure(figsize=(9, 5.5), layout="constrained")
axis = figure.subplots()
for index, n in enumerate(sizes):
    selected = [
        row for row in summary if row["model"] == "cm_multigraph" and row["n"] == n
    ]
    draw_measurements(axis, selected, f"C{index}", f"CM multigraph; n={n:,}")
draw_theory(axis)
core_axes(axis)
axis.set_title(
    "CM multigraph versus theory\nShading: middle half of graphs; bars: one standard error of mean"
)
axis.legend()
figures["validation"] = figure
display(figure)

## Check finite-size agreement away from the critical window

The next table uses the same saved lambda grid for each size. It reports the
root mean squared error and largest absolute difference between a simulated
mean and the limiting curve, outside the fixed exclusion band. This is a
descriptive summary of this grid, not a convergence rate or statistical test.
Individual parameter values can move closer or farther from the prediction
because of Monte Carlo error; monotone improvement is not guaranteed.

We retain signed bias and the standard error at every point in `summary.csv`.
If the curves disagree substantially away from the rise, investigate degree
sampling, graph conventions, peeling, and the chosen support-equation root
before interpreting that difference as a mathematical effect.

In [ ]:
validation_errors = []
for group in summary:
    prediction = cm_prediction(group["lambda"], critical=critical)["rho"]
    group["cm_theory"] = prediction
    group["difference_from_cm_theory"] = group["mean"] - prediction
for n in sizes:
    selected = [
        row
        for row in summary
        if row["model"] == "cm_multigraph"
        and row["n"] == n
        and abs(row["lambda"] - critical["lambda"]) > config["critical_exclusion"]
    ]
    if selected:
        errors = np.array([row["difference_from_cm_theory"] for row in selected])
        validation_errors.append(
            {
                "n": n,
                "lambda_points": len(selected),
                "excluded_half_width": config["critical_exclusion"],
                "rmse": float(np.sqrt(np.mean(errors**2))),
                "mean_absolute_error": float(np.mean(abs(errors))),
                "maximum_absolute_error": float(np.max(abs(errors))),
            }
        )
for result in validation_errors:
    print(
        f"n={result['n']:6,d}: RMSE={result['rmse']:.5f}; "
        f"max absolute difference={result['maximum_absolute_error']:.5f}; "
        f"{result['lambda_points']} lambda points"
    )

## Compare triangle graphs with both configuration-model conventions

Each panel uses the same size, lambda grid, and number of trials. The black
curve remains the **CM** prediction; it is not a derived formula for the triangle
graph. Equal limiting degree laws do not imply equal local neighbourhoods.
The triangle graph groups edges into communities, while CM pairing separates
them in its locally tree-like limit.

The broad view compares core sizes throughout the range. The closer view
keeps all sampled points around the CM critical value, including the earlier
triangle rise. Inspect how the gap and the width of the rise change with $n$.

In [ ]:
for name, zoom in [("models", False), ("zoom", True)]:
    figure = Figure(figsize=(5 * len(sizes), 4.8), layout="constrained")
    axes = figure.subplots(1, len(sizes), squeeze=False)[0]
    for axis, n in zip(axes, sizes):
        for model, label in MODEL_LABELS.items():
            selected = [
                row for row in summary if row["model"] == model and row["n"] == n
            ]
            draw_measurements(
                axis,
                selected,
                MODEL_COLORS[model],
                label,
                "--" if model == "cm_erased" else "-",
            )
        draw_theory(axis)
        core_axes(axis)
        axis.set_title(f"n={n:,}")
        if zoom:
            axis.set_xlim(
                critical["lambda"] - config["focus_half_width"],
                critical["lambda"] + config["focus_half_width"],
            )
            axis.set_ylim(-0.01, 0.6)
        axis.legend(fontsize=8)
    figure.suptitle(
        (
            "Closer view of the rises"
            if zoom
            else "Same limiting degrees, different graph structure"
        )
        + "\nShading: middle half of graphs; bars: one standard error of mean"
    )
    figures[name] = figure
    display(figure)

## Inspect individual graph outcomes rather than only their mean

These histograms give the fraction of trials in each core-fraction bin. All
models use the same bins. A cluster near zero and another at a positive value
would show a mixture of outcomes hidden by the mean. Bins near zero may also
include small nonempty cores, so that bin is not an exact emptiness test.

For the CM, the discontinuity is established by the equations in notebook 05.
For the triangle graph, a sharp finite-size rise or two groups of outcomes
would be evidence to investigate further. These plots alone do not establish
an asymptotic critical value or prove a discontinuity; that requires the
triangle-specific cavity analysis and further finite-size study.

In [ ]:
histogram_n = sizes[-1] if config["histogram_n"] is None else config["histogram_n"]
if histogram_n not in sizes:
    raise ValueError(f"No saved size n={histogram_n}")
histogram_lambdas = config["histogram_lambdas"]
if not histogram_lambdas or any(lam not in lambdas for lam in histogram_lambdas):
    raise ValueError("histogram_lambdas must select saved lambda values")
figure = Figure(figsize=(5 * len(histogram_lambdas), 4.2), layout="constrained")
axes = figure.subplots(1, len(histogram_lambdas), squeeze=False)[0]
for axis, lam in zip(axes, histogram_lambdas):
    for model, label in MODEL_LABELS.items():
        values = [
            row["core_fraction"]
            for row in rows
            if (row["model"], row["n"], row["lambda"]) == (model, histogram_n, lam)
        ]
        axis.hist(
            values,
            bins=np.linspace(0, 1, 41),
            weights=np.ones(len(values)) / len(values),
            histtype="step",
            lw=2,
            color=MODEL_COLORS[model],
            label=label,
            linestyle="--" if model == "cm_erased" else "-",
        )
    axis.set(
        xlabel="3-core fraction",
        ylabel="Fraction of trials in bin",
        title=rf"$\lambda={lam:g}$",
        xlim=(0, 1),
        ylim=(0, 1.04),
    )
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)
figure.suptitle(f"Individual graph outcomes; n={histogram_n:,}")
figures["distributions"] = figure
display(figure)

## Verify the degree-law comparison and measure clustering

Matching only the mean degree would not be enough. For $K=2D$ with
$D\sim\operatorname{Poisson}(\lambda)$, the limiting distribution is


$$
\mathbb{P}(K=2j)=e^{-\lambda}\frac{\lambda^j}{j!},\qquad
\mathbb{P}(K\text{ odd})=0,\qquad
\mathbb{E} K=2\lambda,\quad \operatorname{Var}(K)=4\lambda.
$$


The plot compares the full degree probabilities, including isolated vertices.
CM multigraph degrees follow this law directly. Triangle memberships are
conditioned on the total being divisible by three, and their projection can
merge neighbours; both matter at finite $n$. CM erasure also changes degrees.
For fixed positive $\lambda$ these differences in the empirical degree law
vanish in the sparse limit. We check their size here rather than assuming
finite graphs have identical degree sequences.

The printed clustering statistic is global transitivity on the **simple
projection** in all three cases. Its marked difference helps describe the
structural comparison. It does not control for every correlation created by
triangle communities, so it cannot establish that clustering alone causes
the observed core difference.

In [ ]:
degree_lambda = config["degree_lambda"]
if degree_lambda not in lambdas:
    raise ValueError("degree_lambda must be a saved lambda value")
degree_rows = [row for row in rows if row["lambda"] == degree_lambda]
maximum_degree = max(len(row["degree_histogram"]) - 1 for row in degree_rows)
degrees = np.arange(maximum_degree + 1)
probabilities = np.array(
    [
        exp(-degree_lambda) * degree_lambda ** (k // 2) / factorial(k // 2)
        if k % 2 == 0
        else 0.0
        for k in degrees
    ]
)
figure = Figure(figsize=(5 * len(sizes), 4.2), layout="constrained")
axes = figure.subplots(1, len(sizes), squeeze=False)[0]
for axis, n in zip(axes, sizes):
    for model, label in MODEL_LABELS.items():
        selected = [
            row for row in degree_rows if row["n"] == n and row["model"] == model
        ]
        counts = sum(
            (
                np.pad(
                    row["degree_histogram"],
                    (0, len(degrees) - len(row["degree_histogram"])),
                )
                for row in selected
            ),
            start=np.zeros(len(degrees)),
        )
        axis.plot(
            degrees,
            counts / (n * len(selected)),
            marker="o",
            ms=3,
            color=MODEL_COLORS[model],
            ls="--" if model == "cm_erased" else "-",
            label=label,
        )
    axis.plot(degrees, probabilities, "kx", ms=6, label="2 Poisson(lambda) law")
    axis.set(
        xlabel="Degree (including zero)",
        ylabel="Fraction of vertices",
        title=f"n={n:,}",
    )
    axis.grid(alpha=0.2)
    axis.legend(fontsize=8)
figure.suptitle(rf"Degree-law check at $\lambda={degree_lambda:g}$")
figures["degrees"] = figure
display(figure)
print("Model             n      mean    variance  odd fraction  transitivity")
for group in summary:
    if group["lambda"] == degree_lambda:
        print(
            f"{group['model']:15s} {group['n']:6,d}  {group['degree_mean']:8.4f} "
            f"{group['degree_variance']:9.4f}  {group['odd_degree_fraction']:12.5f} "
            f"{group['simple_transitivity']:12.5f}"
        )
print(f"Limiting mean = {2 * degree_lambda:g}; variance = {4 * degree_lambda:g}.")

## Isolate the effect of erasing loops and repeated edges

Subtracting the two CM results **within each pairing** isolates the effect of
erasure. Averages of these paired differences use their own standard errors;
subtracting two independent-error formulas would discard the pairing.
The difference is nonnegative: erasing edges cannot create new core vertices.

At a fixed lambda the number of loops and repeated edges is typically small,
but this fact alone does not prove that the cores agree. Near a transition,
removing a few edges can trigger a long deletion cascade. The plot therefore
measures the effect directly, including rare large changes. This erased graph
is also different from sampling the configuration model conditioned on being
simple; the simple-graph extension of the theorem does not directly apply to
the erased sampling rule.

In [ ]:
trials = defaultdict(dict)
for row in rows:
    trials[(row["n"], row["lambda"], row["replicate"])][row["model"]] = row
convention_groups, difference_groups = defaultdict(list), defaultdict(list)
for (n, lam, replicate), trial in trials.items():
    if set(trial) != set(MODEL_LABELS):
        raise ValueError(f"Incomplete three-model trial: {(n, lam, replicate)}")
    retained, erased = trial["cm_multigraph"], trial["cm_erased"]
    if (
        retained["seed"] != erased["seed"]
        or retained["trial_seed"] != erased["trial_seed"]
    ):
        raise ValueError("The CM convention control must use the same pairing")
    difference = retained["core_fraction"] - erased["core_fraction"]
    if difference < -1e-12:
        raise ValueError("Erasing edges cannot increase the 3-core")
    convention_groups[(n, lam)].append(difference)
    for model in ("cm_multigraph", "cm_erased"):
        difference_groups[(model, n, lam)].append(
            trial["triangle_rig"]["core_fraction"] - trial[model]["core_fraction"]
        )
convention_differences = [
    {
        "n": n,
        "lambda": lam,
        **sample_statistics(values),
        "fraction_changed": float(np.mean(np.array(values) > 0)),
    }
    for (n, lam), values in sorted(convention_groups.items())
]
model_differences = [
    {
        "comparison": f"triangle_rig_minus_{model}",
        "n": n,
        "lambda": lam,
        **sample_statistics(values),
    }
    for (model, n, lam), values in sorted(difference_groups.items())
]
figure = Figure(figsize=(9, 4.5), layout="constrained")
axis = figure.subplots()
for index, n in enumerate(sizes):
    selected = [row for row in convention_differences if row["n"] == n]
    axis.errorbar(
        [row["lambda"] for row in selected],
        [row["mean"] for row in selected],
        yerr=[row["standard_error"] or 0 for row in selected],
        marker="o",
        ms=3,
        color=f"C{index}",
        capsize=2,
        label=f"n={n:,}",
    )
axis.axvline(critical["lambda"], color="black", ls=":", alpha=0.6)
axis.set(
    xlabel=r"$\lambda$",
    ylabel="CM multigraph minus CM erased: core fraction",
    title="Paired erasure effect\nBars: one standard error of the paired mean difference",
)
axis.grid(alpha=0.25)
axis.legend()
figures["conventions"] = figure
display(figure)

## Describe where a substantial core becomes common

For each model and size, the table brackets the first sampled lambda where at
least half the trials have a core occupying at least `large_core_cutoff` of the
vertices. The lower endpoint is the preceding grid point; no interpolation is
used. A missing endpoint means the grid did not cover that side of the crossing.
The probabilities can fluctuate with sampling, and this is the first observed
crossing rather than a monotonic fit.

This operational summary can show a separation between the models on the
same grid. It is **not** a confidence interval or a mathematically justified
estimate of either asymptotic critical point. In particular, if a model's
limiting density grows continuously from zero, any fixed positive cutoff shifts
this summary above its true critical point. The CM critical point comes from
the support equation; the triangle cavity prediction and its validation are developed in notebooks 08 and 10.

In [ ]:
onset_brackets = []
for model in MODEL_LABELS:
    for n in sizes:
        selected = sorted(
            [row for row in summary if row["model"] == model and row["n"] == n],
            key=lambda row: row["lambda"],
        )
        crossing = next(
            (
                i
                for i, row in enumerate(selected)
                if row["large_core_probability"] >= 0.5
            ),
            None,
        )
        lower = (
            selected[crossing - 1]["lambda"]
            if crossing is not None and crossing > 0
            else None
        )
        upper = selected[crossing]["lambda"] if crossing is not None else None
        onset_brackets.append(
            {
                "model": model,
                "n": n,
                "core_fraction_cutoff": config.get("large_core_cutoff", 0.05),
                "lambda_lower": lower,
                "lambda_upper": upper,
            }
        )
        print(f"{model:15s} n={n:6,d}: first 50% crossing bracket = ({lower}, {upper}]")

save_table(summary, figure_dir / "summary.csv")
save_table(validation_errors, figure_dir / "validation_errors.csv")
save_table(model_differences, figure_dir / "model_differences.csv")
save_table(convention_differences, figure_dir / "convention_differences.csv")
save_table(theory_curve, figure_dir / "theory_curve.csv")
save_table([critical], figure_dir / "critical_point.csv")
save_table(onset_brackets, figure_dir / "onset_brackets.csv")
for name, figure in figures.items():
    figure.savefig(figure_dir / f"{name}.png", dpi=160)
print(f"Saved {len(figures)} figures and analysis tables to {figure_dir}.")

## Findings from the default reference run

The study was executed on 9 November 2026 with the settings in notebook 06:
$n=1{,}000,3{,}000,10{,}000$, 18 lambda values, 60 independent trials at each
point, and seed `20261109`. This section records that run, so it remains
readable with cleared cell outputs. If you change the settings, use the tables
and plots above for your new results rather than carrying these numbers over.

**The CM prediction is well supported away from the transition.** Across the
11 grid points outside $|\lambda-\lambda_c|\leq0.08$, the RMSE of the simulated
means drops from $0.00743$ to $0.00208$ to $0.000732$ as size increases.
At $n=10{,}000$ and $\lambda=1.5$, the measured CM core fraction is
$0.34113\pm0.00143$ (mean and one standard error), compared with the prediction
$0.34100$. This is agreement on the sampled grid, not a measured convergence rate.

**The finite-size critical window is still visible.** At $\lambda=1.295$,
just above $\lambda_c\approx1.28772$, 14 of 60 CM graphs at $n=10{,}000$ have
an empty core. The other 46 have average core fraction $0.17839$. Combining
the two groups gives $0.13677$, below the limiting prediction $0.16705$.
At $\lambda=1.285$, just below the threshold, 35 of 60 graphs still have a
core occupying at least 5% of the vertices. The histograms show why means
around the rise should not be interpreted as a smooth limiting branch.

**The triangle graph develops substantial cores earlier in this study.** At
$n=10{,}000$ and $\lambda=1.24$, 46 of 60 triangle graphs have nonempty cores,
while all 60 CM multigraph cores are empty. At $\lambda=1.26$, their mean
fractions are $0.14106\pm0.00250$ and $0.00980\pm0.00479$, respectively.
The descriptive 50% crossing for a core of at least 5% lies in
$(1.20,1.24]$ for the triangle graph and $(1.275,1.285]$ for CM at this size.
These are sampled-grid brackets, not confidence intervals for critical values.
The triangle histograms near onset have zero and positive outcomes, consistent
with an abrupt transition; this finite-size evidence does not prove its order.

**Earlier emergence does not mean a larger core everywhere.** At
$\lambda=1.5$ and $n=10{,}000$, the triangle mean is $0.32456$, below the CM
mean $0.34113$ by about 1.66 percentage points. Their degree laws are close:
means $2.9961$ and $2.9976$, and variances $5.9912$ and $5.9890$, against targets
3 and 6. Simple-projection transitivity differs substantially, about $0.2505$
for triangles and $0.00054$ for CM. This supports a role for community structure
beyond the degree law, while leaving its different correlations intertwined.

**Erasure matters most near onset.** Its largest mean effect over the grid
falls from $0.01852$ to $0.01092$ to $0.00557$ across the three sizes. At
$n=10{,}000$ and $\lambda=1.295$, one pairing loses $0.1635$ of all vertices
from its core after erasure. At $\lambda=1.5$, the average loss is only
$0.000365$. Sparse defects can therefore have a substantial effect on an
individual graph near criticality, even when their average effect elsewhere
is small.


## What this comparison can establish

Agreement of CM measurements with its predicted curve, improving degree-law
agreement, and a small paired erasure effect away from the transition together
support the implementation and comparison. Read the transition distributions
alongside the means: finite-size rounding can persist where an asymptotic
curve has a jump.

An earlier triangle rise, if it remains separated as $n$ grows, shows that the
degree distribution alone does not determine the 3-core. Triangle communities
change the joint pattern of support between neighbours. Global transitivity
captures part of this structure, but the experiment does not vary clustering
while holding all other local correlations fixed. It therefore supports a
structural explanation, not attribution to a single scalar clustering measure.

The CM jump and critical value are mathematical predictions. Statements about
the triangle transition in this notebook are empirical. Notebook 08 derives the
triangle cavity prediction, and notebook 10 tests it against these measurements
and the focused simulations in notebook 09. That prediction remains a heuristic
for our ensemble rather than a rigorous triangle-core theorem. Increasing repetitions
reduces Monte Carlo uncertainty; increasing graph sizes addresses a different
question, finite-size bias.

Theory and assumptions are derived in notebook 05, using
[Janson and Łuczak (2007), Theorems 2.3 and 2.4](https://arxiv.org/pdf/math/0508453).
The triangle degree-law comparison follows
[van der Hofstad, Komjáthy and Vadon (2021), Corollary 2.9](https://arxiv.org/pdf/1809.02514).